In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

**Q1. How many page lessons**

In [7]:
len(documents)

72

**Q2. Indexing and searching**

In [8]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query,
    boost_dict={"query": 1.0},
    filter_dict={},
    
)

search_results

print(f"First result filename: {search_results[0]['filename']}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


**Q3. RAG**

In [11]:
from rag_helper_homework import RAGBase

rag_assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

query = "How does the agentic loop keep calling the model until it stops?"

result = rag_assistant.rag(query)

print(f"Answer: {result['answer']}")
print(f"Input tokens: {result['input_tokens']}")
print(f"Output tokens: {result['output_tokens']}")

Answer: The loop keeps calling the model by checking whether the response contains any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps going until the model returns a final message with no more tool calls.
Input tokens: 10290
Output tokens: 113


**Q4. Chunking**

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


**Q5. RAG with chunking**

In [16]:
from minsearch import Index

chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

chunk_index.fit(chunks)

# Create RAG with chunk index
rag_chunked = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
)

# Run the same query
result_chunked = rag_chunked.rag(query)

print(f"Answer: {result_chunked['answer']}")
print(f"Input tokens (chunked): {result_chunked['input_tokens']}")
print(f"Input tokens (full): {result['input_tokens']}")

# Calculate reduction
reduction_factor = result['input_tokens'] / result_chunked['input_tokens']
print(f"\nToken reduction factor: {reduction_factor:.1f}x fewer tokens")

Answer: It keeps calling the model inside a `while True` loop, and after each model response it checks whether there were any `function_call` items.

- If the model returns one or more function calls, the code runs them, appends the outputs to `messages`, and loops again.
- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls on that turn**.
Input tokens (chunked): 4734
Input tokens (full): 10290

Token reduction factor: 2.2x fewer tokens


**Q6. Turning it into an agent**

In [49]:
def search(query: str) -> dict[str, str]:
    """
    Search the course materials using the chunk index.
    
    Args:
        query: The search query string
        
    Returns:
        List of relevant documents from the course materials
    """
    results = index.search(query, boost_dict={'question': 2.0})
    return results

In [ ]:
instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.""".strip()

question = "How does the agentic loop work, and how is it different from plain RAG?"



In [51]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# Set up agent instructions

agent_tools = Tools()

agent_tools.add_tool(search)


In [52]:
chat_interface = IPythonChatInterface()
rag_client = OpenAIClient(model='gpt-5.4-mini')
callback = DisplayingRunnerCallback(chat_interface)


rag_client.client = openai_client

runner = OpenAIResponsesRunner(
    tools =agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=rag_client   
)



In [53]:
# print("Launching the ToyAIKit Framework Loop...\n")
# result = runner.loop(prompt=question)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
